# فحص تعليق الانحرافات مقابل الخلايا · أكتوبر 2026

يقرأ نص التعليق، ويستخرج كل رقم يتبعه عنوان خلية مثل `41,300 ريال [Variance!E7]`، ويقارنه بقيمة الخلية
في `variance_oct.xlsx`. ثم يسرد الأرقام التي لا خلية لها. لا يحكم على الأسباب: هذا لك.

In [ ]:
# الإعداد: يجد الملفات بجوار الدفتر أو في المجلد الأعلى. في خدمة دفاتر مستضافة: ارفع الملفات بجوار الدفتر.
import csv, datetime, hashlib
from pathlib import Path
import pandas as pd

def find(name):
    for p in (Path("."), Path(".."), Path("files")):
        if (p / name).exists():
            return p / name
    raise FileNotFoundError(name + " : ضع الملف بجوار الدفتر")

def log_run(notebook, inputs, totals):
    """يضيف سطرًا إلى سجل التشغيل: متى، وعلى أي ملفات (بصمتها)، وبأي مجاميع رقابية."""
    stamp = datetime.datetime.now().isoformat(timespec="seconds")
    marks = ";".join(p.name + ":" + hashlib.sha256(p.read_bytes()).hexdigest()[:12] for p in inputs)
    new = not Path("run_log.csv").exists()
    with open("run_log.csv", "a", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["run_at", "notebook", "inputs_sha256", "control_totals"])
        w.writerow([stamp, notebook, marks, totals])
    print("سُجّل التشغيل في run_log.csv")

In [ ]:
import re
from openpyxl import load_workbook
COMMENTARY = "ai_draft_commentary.txt"     # غيّره إلى ملف تعليقك، مثل my_commentary.txt
text_path, book_path = find(COMMENTARY), find("variance_oct.xlsx")
text = text_path.read_text(encoding="utf-8")
sheet = load_workbook(book_path, data_only=True)["Variance"]
TAG = re.compile(r"(\d[\d,]*(?:\.\d+)?)\s*(%?)\s*(?:ريال)?\s*\[Variance!([A-F]\d+)\]")
rows = []
for num, pct, cell in TAG.findall(text):
    stated = float(num.replace(",", ""))
    value = sheet[cell].value
    if value is None:
        raise SystemExit("الخلية " + cell + " بلا قيمة محفوظة: افتح الملف في جدول البيانات واحفظه ثم أعد التشغيل")
    shown = round(abs(value) * 100, 1) if pct else round(abs(value))
    rows.append({"cell": cell, "stated": stated, "in_cell": shown, "match": stated == shown})
tags = pd.DataFrame(rows)
print(tags.to_string(index=False))
print("أرقام لها خلية:", len(tags), "· مطابقة:", int(tags["match"].sum()), "· غير مطابقة:", int((~tags["match"]).sum()))

In [ ]:
# أرقام بفواصل آلاف لا يتبعها عنوان خلية
untagged = [n for n in re.findall(r"\d{1,3}(?:,\d{3})+(?![\d,])(?!\s*(?:ريال)?\s*\[Variance!)", text)]
print("أرقام بلا خلية:", untagged)
# المراكز التي تستوجب التعليق حسب القاعدة: 20,000 ريال و10% معًا
material = [sheet[f"A{r}"].value for r in range(4, 16)
            if abs(sheet[f"E{r}"].value) >= 20000 and abs(sheet[f"F{r}"].value) >= 0.10]
missing = [c for c in material if c not in text]
extra = [sheet[f"A{r}"].value for r in range(4, 16) if sheet[f"A{r}"].value in text and sheet[f"A{r}"].value not in material]
print("تستوجب التعليق:", material)
print("غائبة عن النص:", missing, "· معلَّق عليها وهي دون القاعدة:", extra)
log_run("commentary_check", [text_path, book_path],
        f"tags {len(tags)}; match {int(tags['match'].sum())}; untagged {len(untagged)}; missing {len(missing)}")